# Graph Autoencoder

An MLP-based autoencoder for reconstructing random graphs from their adjacency matrices.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

## Random Graph Generation

In [ ]:
def generate_random_graph(n=20, p=0.5):
    """Generate a random undirected graph adjacency matrix.
    
    Args:
        n: Number of nodes.
        p: Probability of an edge existing between any two nodes.
    
    Returns:
        adj: (n, n) symmetric adjacency matrix with zeros on the diagonal.
    """
    upper = np.random.binomial(1, p, size=(n, n))
    # Make symmetric and zero diagonal
    adj = np.triu(upper, k=1)
    adj = adj + adj.T
    return adj.astype(np.float32)


def plot_adjacency(adj, title="Adjacency Matrix"):
    plt.figure(figsize=(5, 5))
    plt.imshow(adj, cmap="Blues", vmin=0, vmax=1)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.show()


# Example
adj = generate_random_graph(n=20, p=0.5)
plot_adjacency(adj)

## MLP Autoencoder

Architecture: `400 → 256 → 128 → 64 → 32 → 64 → 128 → 256 → 400`

The input is the flattened adjacency matrix (20×20 = 400 values).

In [ ]:
class GraphAutoencoder(nn.Module):
    def __init__(self, n=20):
        super().__init__()
        input_dim = n * n  # 400

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128),       nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, 32),         nn.ReLU(),
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),         nn.ReLU(),
            nn.Linear(64, 128),        nn.ReLU(),
            nn.Linear(128, 256),       nn.ReLU(),
            nn.Linear(256, input_dim), nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

    def encode(self, x):
        return self.encoder(x)


model = GraphAutoencoder(n=20)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## Dataset & Training

In [ ]:
# Generate a dataset of random graphs
N_GRAPHS = 2000
N_NODES  = 20
EDGE_P   = 0.5

graphs = np.stack([generate_random_graph(N_NODES, EDGE_P).flatten()
                   for _ in range(N_GRAPHS)])
dataset = torch.tensor(graphs)  # (N_GRAPHS, 400)

# Train / val split
split = int(0.8 * N_GRAPHS)
train_data = dataset[:split]
val_data   = dataset[split:]

print(f"Train: {train_data.shape}, Val: {val_data.shape}")

In [ ]:
EPOCHS     = 100
BATCH_SIZE = 64
LR         = 1e-3

optimizer  = optim.Adam(model.parameters(), lr=LR)
criterion  = nn.BCELoss()

train_losses = []
val_losses   = []

for epoch in range(1, EPOCHS + 1):
    # --- training ---
    model.train()
    idx = torch.randperm(len(train_data))
    epoch_loss = 0.0
    for i in range(0, len(train_data), BATCH_SIZE):
        batch = train_data[idx[i:i + BATCH_SIZE]]
        recon = model(batch)
        loss  = criterion(recon, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(batch)
    train_losses.append(epoch_loss / len(train_data))

    # --- validation ---
    model.eval()
    with torch.no_grad():
        val_recon = model(val_data)
        val_loss  = criterion(val_recon, val_data).item()
    val_losses.append(val_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train {train_losses[-1]:.4f} | val {val_losses[-1]:.4f}")

In [ ]:
plt.plot(train_losses, label="train")
plt.plot(val_losses,   label="val")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Autoencoder Training")
plt.legend()
plt.tight_layout()
plt.show()

## Reconstruction Quality

In [ ]:
model.eval()
with torch.no_grad():
    sample    = val_data[0].unsqueeze(0)
    recon_raw = model(sample).squeeze().numpy()

original = val_data[0].numpy().reshape(N_NODES, N_NODES)
recon    = recon_raw.reshape(N_NODES, N_NODES)
# Threshold at 0.5 to get a binary graph
recon_bin = (recon >= 0.5).astype(np.float32)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, mat, title in zip(
    axes,
    [original, recon, recon_bin],
    ["Original", "Reconstruction (raw)", "Reconstruction (binary)"],
):
    ax.imshow(mat, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(title)
plt.tight_layout()
plt.show()

accuracy = (recon_bin == original).mean()
print(f"Edge-level accuracy: {accuracy:.4f}")